In [ ]:
import torch
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

from simple_unet import SimpleUnet, SimpleUnetWithTime


## モデルの動作確認

In [ ]:
model_unet = SimpleUnet()
x = torch.randn(10,1,28,28)
y = model_unet(x)
y.shape

In [ ]:
model_unet_with_tiem = SimpleUnetWithTime()
x = torch.randn(10,1,28,28)
timestep = torch.randn(10, 1)
y = model_unet_with_tiem(x, timestep)
y.shape

## 画像の処理

In [ ]:
# load image
# current_dir = os.path.dirname(os.path.abspath(__file__))
# file_path = os.path.join(current_dir, 'flower.png')
file_path = r"C:\Users\testuser\Desktop\技術研修\技術研修_モデル\flower.png"
image = plt.imread(file_path)
print(image.shape)  # (64, 64, 3)

# preprocess
preprocess = transforms.ToTensor()
x = preprocess(image)
print(x.shape)  # (3, 64, 64)

original_x = x.clone()  # keep original image

In [ ]:
def reverse_to_img(x):
    x = x * 255
    x = x.clamp(0, 255)
    x = x.to(torch.uint8)
    to_pil = transforms.ToPILImage()
    return to_pil(x)

### タイムステップごとの拡散過程

In [ ]:
T = 500
beta_start = 0.0001
beta_end = 0.02
betas = torch.linspace(beta_start, beta_end, T)
imgs = []

for t in range(T):
    if t % 100 == 0:
        img = reverse_to_img(x)
        imgs.append(img)

    beta = betas[t]
    eps = torch.randn_like(x)
    x = torch.sqrt(1 - beta) * x + torch.sqrt(beta) * eps

# show imgs
plt.figure(figsize=(15, 6))
for i, img in enumerate(imgs[:10]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(img)
    plt.title(f'Noise: {i * 100}')
    plt.axis('off')
plt.show()


### $q(x_t|x_0)$からのサンプリング

In [ ]:
def add_noise(x_0, timestep, betas):
    T = len(betas)
    assert t >=1 and t<=T

    alphas = 1 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    time_idx = timestep - 1  # リストが0から始まるため
    alpha_bar = alpha_bars[time_idx]

    eps = torch.randn_like(x_0)
    x_t = torch.sqrt(alpha_bar) * x_0 + torch.sqrt(1 - alpha_bar) * eps

    return x_t


In [ ]:
# 好きな時刻のノイズ画像を出力できる
t = 125
x_t = add_noise(original_x, t, betas)

img = reverse_to_img(x_t)
plt.imshow(img)
plt.title(f'Noise: {t}')
plt.axis('off')
plt.show()